In [1]:
pip install numpy pandas matplotlib scikit-learn jupyter tensorflow keras seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ──────────────────────────────────────────────
# 1. IMPORTS
# ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")


TensorFlow version : 2.20.0
Keras version      : 3.12.0


In [3]:
# ──────────────────────────────────────────────
# 2. CLASS LABELS
# ──────────────────────────────────────────────
CLASS_NAMES = [
    "T-shirt/top",   # 0
    "Trouser",       # 1
    "Pullover",      # 2
    "Dress",         # 3
    "Coat",          # 4
    "Sandal",        # 5
    "Shirt",         # 6
    "Sneaker",       # 7
    "Bag",           # 8
    "Ankle boot",    # 9
]
NUM_CLASSES = len(CLASS_NAMES)

In [4]:
# ──────────────────────────────────────────────
# 3. LOAD & PRE-PROCESS DATA
# ──────────────────────────────────────────────
def load_and_preprocess():
    """Load Fashion-MNIST, normalise, reshape, one-hot encode."""
    print("\n[1/5] Loading Fashion-MNIST …")
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

    # Pixel values → [0, 1]
    x_train = x_train.astype("float32") / 255.0
    x_test  = x_test.astype("float32")  / 255.0

    # Add channel dimension: (N, 28, 28) → (N, 28, 28, 1)
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[...,  np.newaxis]

    # One-hot encode labels
    y_train_oh = to_categorical(y_train, NUM_CLASSES)
    y_test_oh  = to_categorical(y_test,  NUM_CLASSES)

    print(f"   Train : {x_train.shape}  Labels : {y_train_oh.shape}")
    print(f"   Test  : {x_test.shape}   Labels : {y_test_oh.shape}")
    return x_train, y_train, y_train_oh, x_test, y_test, y_test_oh

In [5]:
# ──────────────────────────────────────────────
# 4. DATA AUGMENTATION
# ──────────────────────────────────────────────
def build_augmentation_layer():
    """
    Keras sequential preprocessing layer for on-the-fly augmentation
    applied only during training.
    """
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    ], name="augmentation")

In [6]:
# ──────────────────────────────────────────────
# 5. BUILD THE CNN MODEL
# ──────────────────────────────────────────────
def build_model(input_shape=(28, 28, 1)):
    """
    Architecture
    ─────────────
    Input → Augmentation (train only)
    → Conv2D(32) + BN + ReLU  → MaxPool → Dropout
    → Conv2D(64) + BN + ReLU  → MaxPool → Dropout
    → Conv2D(128)+ BN + ReLU  → MaxPool → Dropout
    → Flatten
    → Dense(256) + BN + ReLU  → Dropout
    → Dense(128) + BN + ReLU  → Dropout
    → Dense(10, softmax)
    """
    print("\n[2/5] Building CNN model …")

    inputs = keras.Input(shape=input_shape, name="input")

    # ── Augmentation (training only) ──
    aug = build_augmentation_layer()
    x = aug(inputs, training=True)   # Keras passes training flag automatically

    # ── Block 1 ──
    x = layers.Conv2D(32, (3, 3), padding="same",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv1_1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, (3, 3), padding="same",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv1_2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 2 ──
    x = layers.Conv2D(64, (3, 3), padding="same",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv2_1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), padding="same",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv2_2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 3 ──
    x = layers.Conv2D(128, (3, 3), padding="same",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv3_1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Classifier head ──
    x = layers.Flatten()(x)
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)

    model = keras.Model(inputs, outputs, name="FashionCNN")
    model.summary()
    return model

In [7]:
# ──────────────────────────────────────────────
# 6. COMPILE & TRAIN
# ──────────────────────────────────────────────
def compile_and_train(model, x_train, y_train_oh, x_test, y_test_oh):
    print("\n[3/5] Compiling & training …")

    # ── Learning-rate schedule ──
    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=10,
        t_mul=2.0,
        m_mul=0.9,
        alpha=1e-5,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    # ── Callbacks ──
    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_accuracy", patience=10,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ModelCheckpoint(
            "best_fashion_cnn.keras",
            monitor="val_accuracy", save_best_only=True, verbose=1
        ),
    ]

    history = model.fit(
        x_train, y_train_oh,
        batch_size=64,
        epochs=50,
        validation_data=(x_test, y_test_oh),
        callbacks=cb_list,
        verbose=1,
    )
    return history


In [8]:
# ──────────────────────────────────────────────
# 7. EVALUATE
# ──────────────────────────────────────────────
def evaluate_model(model, x_test, y_test, y_test_oh):
    print("\n[4/5] Evaluating …")
    loss, acc = model.evaluate(x_test, y_test_oh, verbose=0)
    print(f"\n   Test Loss     : {loss:.4f}")
    print(f"   Test Accuracy : {acc * 100:.2f}%")

    y_pred_prob = model.predict(x_test, verbose=0)
    y_pred      = np.argmax(y_pred_prob, axis=1)

    print("\n   Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    return y_pred, y_pred_prob


In [9]:
# ──────────────────────────────────────────────
# 8. VISUALISE RESULTS
# ──────────────────────────────────────────────
def plot_all(history, x_test, y_test, y_pred, y_pred_prob):
    print("\n[5/5] Generating plots …")
    plt.style.use("seaborn-v0_8-darkgrid")

    # ── 8a. Training curves ──────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training History", fontsize=16, fontweight="bold")

    axes[0].plot(history.history["accuracy"],     label="Train Acc")
    axes[0].plot(history.history["val_accuracy"], label="Val Acc")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history.history["loss"],     label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Val Loss")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()
    print("   Saved → training_curves.png")

    # ── 8b. Confusion Matrix ─────────────────────
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, ax=ax
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title("Confusion Matrix — Fashion MNIST CNN", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.show()
    print("   Saved → confusion_matrix.png")

    # ── 8c. Sample predictions grid ──────────────
    n_show = 30
    indices = np.random.choice(len(x_test), n_show, replace=False)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle("Sample Predictions (green = correct, red = wrong)",
                 fontsize=14, fontweight="bold")

    for i, idx in enumerate(indices):
        ax = fig.add_subplot(3, 10, i + 1)
        ax.imshow(x_test[idx].squeeze(), cmap="gray")
        ax.axis("off")

        true_label = CLASS_NAMES[y_test[idx]]
        pred_label = CLASS_NAMES[y_pred[idx]]
        conf       = y_pred_prob[idx][y_pred[idx]] * 100

        color = "green" if y_test[idx] == y_pred[idx] else "red"
        ax.set_title(f"{pred_label}\n{conf:.0f}%", fontsize=7,
                     color=color, pad=2)

    plt.tight_layout()
    plt.savefig("sample_predictions.png", dpi=150)
    plt.show()
    print("   Saved → sample_predictions.png")

    # ── 8d. Per-class accuracy bar chart ─────────
    cm_norm   = cm.astype("float") / cm.sum(axis=1, keepdims=True)
    class_acc = np.diag(cm_norm) * 100

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(CLASS_NAMES, class_acc,
                  color=plt.cm.RdYlGn(class_acc / 100))
    ax.set_ylim([0, 110])
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Per-Class Accuracy", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right")

    for bar, val in zip(bars, class_acc):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1, f"{val:.1f}%",
                ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig("per_class_accuracy.png", dpi=150)
    plt.show()
    print("   Saved → per_class_accuracy.png")

In [10]:
# ──────────────────────────────────────────────
# 9. SINGLE-IMAGE PREDICTION HELPER
# ──────────────────────────────────────────────
def predict_single(model, image, true_label=None):
    """
    Predict one image.
    image : numpy array shape (28, 28) or (28, 28, 1), values in [0, 1]
    """
    if image.ndim == 2:
        image = image[..., np.newaxis]
    img_batch = np.expand_dims(image, axis=0)

    probs = model.predict(img_batch, verbose=0)[0]
    pred  = np.argmax(probs)

    print(f"\n   Predicted class : {CLASS_NAMES[pred]}  ({probs[pred]*100:.1f}%)")
    if true_label is not None:
        print(f"   True class      : {CLASS_NAMES[true_label]}")

    # Bar chart of class probabilities
    plt.figure(figsize=(8, 4))
    bars = plt.barh(CLASS_NAMES, probs * 100,
                    color=["green" if i == pred else "steelblue"
                           for i in range(NUM_CLASSES)])
    plt.xlabel("Probability (%)")
    plt.title("Prediction Confidence")
    plt.xlim([0, 110])
    for bar, p in zip(bars, probs):
        plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f"{p*100:.1f}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("single_prediction.png", dpi=150)
    plt.show()
    return CLASS_NAMES[pred], probs


In [ ]:
# ──────────────────────────────────────────────
# 10. MAIN
# ──────────────────────────────────────────────
def main():
    # Load data
    x_train, y_train, y_train_oh, x_test, y_test, y_test_oh = load_and_preprocess()

    # Build model
    model = build_model(input_shape=(28, 28, 1))

    # Train
    history = compile_and_train(model, x_train, y_train_oh, x_test, y_test_oh)

    # Evaluate
    y_pred, y_pred_prob = evaluate_model(model, x_test, y_test, y_test_oh)

    # Visualise
    plot_all(history, x_test, y_test, y_pred, y_pred_prob)

    # Demo: predict a random test image
    print("\n── Demo: single-image prediction ──")
    rand_idx = np.random.randint(0, len(x_test))
    predict_single(model, x_test[rand_idx], true_label=y_test[rand_idx])

    # Save final model
    model.save("fashion_mnist_cnn_final.keras")
    print("\nModel saved → fashion_mnist_cnn_final.keras")
    print("\nDone! ✓")


if __name__ == "__main__":
    main()



[1/5] Loading Fashion-MNIST …
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 74s 3us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 83us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 4s 1us/step
   Train : (60000, 28, 28, 1)  Labels : (60000, 10)
   Test  : (10000, 28, 28, 1)   Labels : (10000, 10)

[2/5] Building CNN model …


Model: "FashionCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)                   │ (None, 28, 28, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ augmentation (Sequential)            │ (None, 28, 28, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1_1 (Conv2D)                     │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 28, 28, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 28, 28, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1_2 (Conv2D)                     │ (None, 28, 28, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 28, 28, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 28, 28, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2_1 (Conv2D)                     │ (None, 14, 14, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 14, 14, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_2 (Activation)            │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2_2 (Conv2D)                     │ (None, 14, 14, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 14, 14, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_3 (Activation)            │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 7, 7, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 7, 7, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3_1 (Conv2D)                     │ (None, 7, 7, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 471,018 (1.80 MB)

 Trainable params: 469,610 (1.79 MB)

 Non-trainable params: 1,408 (5.50 KB)


[3/5] Compiling & training …
Epoch 1/50
 19/938 ━━━━━━━━━━━━━━━━━━━━ 3:06 203ms/step - accuracy: 0.1079 - loss: 2.9769